# The Hailo stick from Python

Hailo's own Python package (`hailo_platform`, HailoRT 5.3.0): on Linux and Windows the one that comes with HailoRT,
on a Mac the build from the community macOS USB port (`tools/build_hailort_wheel.sh`, see `README.md`).
Choose the kernel that has it installed (here: **Python 3.12 (Hailo stick)**).

Only one program can use the stick at a time: quit Hailo Studio (and close other notebooks using it) first.
Each section opens the stick and releases it at the end, so the sections can be run in any order.
Hailo's own tutorials are next to this one in `hailo_tutorials/` (set their `MODEL_PATH` to a file in `../models`).

In [1]:
from pathlib import Path
import time
import numpy as np
from PIL import Image
from hailo_platform import Device, VDevice, FormatType

MODELS = Path("../models")
print("Sticks found:", Device.scan())

Sticks found: ['usb/000:010']


## 1. Find objects (YOLOv8s)

In [2]:
image = Image.open("../testdata/bus.jpg").convert("RGB")
COCO = {0: "person", 2: "car", 5: "bus", 7: "truck"}

with VDevice() as vdevice:
    model = vdevice.create_infer_model(str(MODELS / "yolov8s.hef"))
    for output in model.outputs:
        model.output(output.name).set_format_type(FormatType.FLOAT32)
    height, width, _ = model.input().shape
    with model.configure() as configured:
        bindings = configured.create_bindings(
            output_buffers={o.name: np.empty(model.output(o.name).shape, dtype=np.float32) for o in model.outputs})
        bindings.input().set_buffer(np.array(image.resize((width, height)), dtype=np.uint8))
        start = time.time()
        for _ in range(20):
            configured.run([bindings], 10_000)
        print(f"{20 / (time.time() - start):.0f} pictures a second")
        per_class = bindings.output().get_buffer()   # one list per COCO class: [y0, x0, y1, x1, score], 0…1

for cls, found in enumerate(per_class):
    for y0, x0, y1, x1, score in found:
        if score > 0.5:
            print(f"{COCO.get(cls, cls):>7}  {score:.2f}  x {x0:.2f}–{x1:.2f}  y {y0:.2f}–{y1:.2f}")

47 pictures a second


 person  0.92  x 0.06–0.30  y 0.37–0.84
 person  0.90  x 0.83–1.00  y 0.36–0.82
 person  0.88  x 0.28–0.43  y 0.38–0.80
    bus  0.89  x 0.00–1.00  y 0.21–0.69


## 2. Chat (Qwen2.5-1.5B), word by word

In [3]:
from hailo_platform.genai import LLM

with VDevice() as vdevice:
    llm = LLM(vdevice, str(MODELS / "Qwen2.5-1.5B-Instruct.hef"), "")
    prompt = [{"role": "system", "content": "You are a helpful assistant. Answer briefly."},
              {"role": "user", "content": "Give me three names for a robot cat."}]
    with llm.generate(prompt, seed=7, max_generated_tokens=120) as generation:
        for token in generation:
            print(token, end="", flush=True)
    llm.release()

1

.

 Wh

isk

er

2

... (7 outputs left out)

ow

W

oo

<|im_end|>

## 3. Ask about a picture (Qwen2-VL-2B)

In [4]:
from hailo_platform.genai import VLM

with VDevice() as vdevice:
    # False: the Mac build of HailoRT has no host-side tokenizer, so the stick tokenizes (True fails with status 7).
    vlm = VLM(vdevice, str(MODELS / "Qwen2-VL-2B-Instruct.hef"), False)
    height, width, _ = vlm.input_frame_shape()
    frame = np.array(image.resize((width, height)), dtype=np.uint8)
    prompt = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "Describe this picture in one sentence."}]}]
    print(vlm.generate_all(prompt, frames=[frame], max_generated_tokens=80))
    vlm.release()

A man in a beige coat and sunglasses stands next to a blue bus with the text "CERO EMISIONES" on it, while another person in a black coat walks by.<|im_end|>


## 4. Speech to text (Whisper-Base)

In [5]:
import wave
from hailo_platform.genai import Speech2Text, Speech2TextTask

with wave.open("../web/samples/speech_sample.wav") as w:   # 16 kHz mono 16-bit
    audio = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16).astype(np.float32) / 32768

with VDevice() as vdevice:
    whisper = Speech2Text(vdevice, str(MODELS / "Whisper-Base.hef"))
    for segment in whisper.generate_all_segments(audio, task=Speech2TextTask.TRANSCRIBE, language="en"):
        print(f"{segment.start_sec:4.1f}–{segment.end_sec:4.1f} s {segment.text}")
    whisper.release()

 0.0– 3.9 s  Hello from the iPad, the quick brown fox jumps over the lazy dog.
